In [52]:
import numpy as np

In [53]:
image = np.array([
    [1, 2, 0],
    [0, 1, 3],
    [2, 0, 1]
], dtype=float)

kernel = np.array([
    [1,  0],
    [-1, 1]
], dtype=float)

kernel_bias = 0.5

weights = np.full((1, 1), 0.8)
bias = np.full(1, 0.5)

learning_rate = 0.01

In [54]:
def relu(x):
    return np.maximum(0, x)


def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))


def max_pooling(x):
    h, w = x.shape
    output = np.zeros((h // 2, w // 2))
    switches = {}

    for i in range(0, h - 1, 2):
        for j in range(0, w - 1, 2):
            patch = x[i:i+2, j:j+2]
            pi, pj = np.unravel_index(
                np.argmax(patch), patch.shape
            )

            output[i // 2, j // 2] = patch[pi, pj]
            switches[(i // 2, j // 2)] = (
                i + pi, j + pj
            )

    return output, switches


In [ ]:
def forward(image):
    h, w = image.shape
    kh, kw = kernel.shape

    conv = np.zeros((h-kh+1, w-kw+1))

    for i in range(conv.shape[0]):
        for j in range(conv.shape[1]):
            patch = image[i:i+kh, j:j+kw]
            conv[i, j] = np.sum(patch * kernel) + kernel_bias

    activated = relu(conv)

    pooled, switches = max_pooling(activated)

    flattened = pooled.flatten()

    logits = weights @ flattened + bias
    output = sigmoid(logits)

    loss = 1 - output[0]

    cache = (
        conv, activated, pooled, switches,
        flattened, logits, output
    )

    return output, loss, cache

In [ ]:
def backward(image, cache):
    global kernel, kernel_bias, weights, bias

    (
        conv, activated, pooled, switches,
        flattened, logits, output
    ) = cache
    dlogits = -output * (1 - output)

    dweights = dlogits.reshape(-1, 1) @ flattened.reshape(1, -1)
    dbias = dlogits.copy()

    dflattened = weights.T @ dlogits

    dpooled = dflattened.reshape(pooled.shape)

    dactivated = np.zeros_like(activated)

    for i in range(pooled.shape[0]):
        for j in range(pooled.shape[1]):
            pi, pj = switches[(i, j)]
            dactivated[pi, pj] += dpooled[i, j]

    dconv = dactivated * (conv > 0)


    dkernel = np.zeros_like(kernel)
    dkernel_bias = np.sum(dconv)

    kh, kw = kernel.shape

    for i in range(dconv.shape[0]):
        for j in range(dconv.shape[1]):
            dkernel += (
                dconv[i, j] * image[i:i+kh, j:j+kw]
            )

    weights -= learning_rate * dweights
    bias -= learning_rate * dbias

    kernel -= learning_rate * dkernel
    kernel_bias -= learning_rate * dkernel_bias

    return dkernel, dkernel_bias, dweights, dbias


In [ ]:
output, loss, cache = forward(image)

print("Convolution output:\n", cache[0])
print("ReLU output:\n", cache[1])
print("Pooling output:\n", cache[2])
print("Flattened output:", cache[4])
print("Sigmoid output:", output)
print("Loss before update:", loss)

dkernel, dkernel_bias, dweights, dbias = backward(
    image, cache
)

print("\nGradients:")
print("Kernel gradient:\n", dkernel)
print("Kernel bias gradient:", dkernel_bias)
print("Weights gradient:\n", dweights)
print("Dense bias gradient:", dbias)

print("\nUpdated parameters:")
print("Kernel:\n", kernel)
print("Kernel bias:", kernel_bias)
print("Weights:\n", weights)
print("Dense bias:", bias)
new_output, new_loss, _ = forward(image)
print("\nOutput after update:", new_output)
print("Loss after update:", new_loss)

Convolution output:
 [[ 2.5  4.5]
 [-1.5  2.5]]
ReLU output:
 [[2.5 4.5]
 [0.  2.5]]
Pooling output:
 [[4.5]]
Flattened output: [4.5]
Sigmoid output: [0.9836975]
Loss before update: 0.016302499371440904

Gradients:
Kernel gradient:
 [[-0.02565876  0.        ]
 [-0.01282938 -0.03848815]]
Kernel bias gradient: -0.01282938230854806
Weights gradient:
 [[-0.07216528]]
Dense bias gradient: [-0.01603673]

Updated parameters:
Kernel:
 [[ 1.00025659  0.        ]
 [-0.99987171  1.00038488]]
Kernel bias: 0.5001282938230854
Weights:
 [[0.80072165]]
Dense bias: [0.50016037]

Output after update: [0.98377667]
Loss after update: 0.01622332777981883
